# Chronic Kidney Disease - Exploratory Data Analysis
**Dataset**: 1,659 samples, 54 features
**Target**: Diagnosis (0=No CKD, 1=CKD)
**Note**: Synthetic dataset with 11:1 class imbalance - will need SMOTE

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Paths
DATA_PATH = '../../datasets/chronic_kidney_disease.csv'
FIGURES_PATH = '../../notebooks/eda/figures/kidney'
os.makedirs(FIGURES_PATH, exist_ok=True)

# Load data
df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
print(f'\nColumns ({len(df.columns)}):')
print(df.columns.tolist())

In [ ]:
# Basic info
print('Data Types:')
print(df.dtypes.value_counts())
print(f'\nMissing Values: {df.isnull().sum().sum()}')
print(f'\nTarget Distribution:')
print(df['Diagnosis'].value_counts())
print(f'\nClass Ratio: {df["Diagnosis"].value_counts()[1] / df["Diagnosis"].value_counts()[0]:.2f}:1')

In [ ]:
# Figure 1: Target Distribution
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#2ecc71', '#e74c3c']
counts = df['Diagnosis'].value_counts().sort_index()
bars = ax.bar(['No CKD (0)', 'CKD (1)'], counts.values, color=colors, edgecolor='black', linewidth=1.5)

for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
            f'{count}\n({count/len(df)*100:.1f}%)', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('Count', fontsize=12)
ax.set_title('Chronic Kidney Disease - Target Distribution\n(11:1 Imbalance - Needs SMOTE)', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(counts.values) * 1.15)
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}/01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Key kidney biomarkers for analysis
kidney_biomarkers = ['SerumCreatinine', 'BUNLevels', 'GFR', 'HemoglobinLevels', 
                     'ProteinInUrine', 'ACR', 'SerumElectrolytesSodium', 'SerumElectrolytesPotassium']

# Additional clinical features
clinical_features = ['Age', 'BMI', 'SystolicBP', 'DiastolicBP', 'FastingBloodSugar', 
                     'HbA1c', 'CholesterolTotal', 'CholesterolLDL', 'CholesterolHDL']

print('Key Kidney Biomarkers Statistics:')
df[kidney_biomarkers].describe().round(2)

In [ ]:
# Figure 2: Key Kidney Biomarkers by Diagnosis
fig, axes = plt.subplots(2, 4, figsize=(16, 10))
axes = axes.flatten()

for idx, col in enumerate(kidney_biomarkers):
    ax = axes[idx]
    for diagnosis, color, label in [(0, '#2ecc71', 'No CKD'), (1, '#e74c3c', 'CKD')]:
        data = df[df['Diagnosis'] == diagnosis][col]
        ax.hist(data, bins=30, alpha=0.6, color=color, label=label, edgecolor='white')
    ax.set_xlabel(col, fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_title(f'{col}', fontsize=11, fontweight='bold')

plt.suptitle('Key Kidney Biomarkers Distribution by Diagnosis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}/02_kidney_biomarkers_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: GFR vs Serum Creatinine (key kidney relationship)
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(df['SerumCreatinine'], df['GFR'], c=df['Diagnosis'], 
                     cmap='RdYlGn_r', alpha=0.6, edgecolor='white', linewidth=0.5)

ax.set_xlabel('Serum Creatinine (mg/dL)', fontsize=12)
ax.set_ylabel('GFR (mL/min/1.73m²)', fontsize=12)
ax.set_title('GFR vs Serum Creatinine by Diagnosis\n(Higher Creatinine + Lower GFR = CKD)', fontsize=14, fontweight='bold')

# Add CKD stage lines
ax.axhline(y=90, color='green', linestyle='--', alpha=0.5, label='Stage 1 (>90)')
ax.axhline(y=60, color='yellow', linestyle='--', alpha=0.5, label='Stage 2 (60-89)')
ax.axhline(y=30, color='orange', linestyle='--', alpha=0.5, label='Stage 3 (30-59)')
ax.axhline(y=15, color='red', linestyle='--', alpha=0.5, label='Stage 4 (15-29)')

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Diagnosis (0=No CKD, 1=CKD)', fontsize=10)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}/03_gfr_vs_creatinine.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 4: Correlation Heatmap - Key Features
key_features = ['Age', 'BMI', 'SystolicBP', 'DiastolicBP', 'FastingBloodSugar', 'HbA1c',
                'SerumCreatinine', 'BUNLevels', 'GFR', 'HemoglobinLevels', 
                'CholesterolTotal', 'CholesterolHDL', 'Diagnosis']

corr_matrix = df[key_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap - Chronic Kidney Disease', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}/04_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 5: Age Distribution by Diagnosis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution
ax = axes[0]
for diagnosis, color, label in [(0, '#2ecc71', 'No CKD'), (1, '#e74c3c', 'CKD')]:
    data = df[df['Diagnosis'] == diagnosis]['Age']
    ax.hist(data, bins=25, alpha=0.6, color=color, label=label, edgecolor='white')
ax.set_xlabel('Age (years)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Age Distribution by Diagnosis', fontsize=12, fontweight='bold')
ax.legend()

# Boxplot comparison
ax = axes[1]
df.boxplot(column=['SerumCreatinine', 'BUNLevels'], by='Diagnosis', ax=ax)
ax.set_title('Kidney Markers by Diagnosis', fontsize=12, fontweight='bold')
ax.set_xlabel('Diagnosis (0=No CKD, 1=CKD)', fontsize=12)
plt.suptitle('')

plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}/05_age_and_kidney_markers.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 6: Comorbidities and Risk Factors
risk_factors = ['Smoking', 'FamilyHistoryKidneyDisease', 'FamilyHistoryHypertension', 
                'FamilyHistoryDiabetes', 'PreviousAcuteKidneyInjury', 'UrinaryTractInfections']

fig, axes = plt.subplots(2, 3, figsize=(14, 10))
axes = axes.flatten()

for idx, col in enumerate(risk_factors):
    ax = axes[idx]
    ct = pd.crosstab(df[col], df['Diagnosis'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='black')
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Percentage (%)')
    ax.set_xticklabels(['No', 'Yes'], rotation=0)
    ax.legend(['No CKD', 'CKD'], fontsize=9)

plt.suptitle('Risk Factors and CKD Diagnosis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}/06_risk_factors.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 7: Feature Distributions for Unified Schema Mapping
unified_features = ['Age', 'BMI', 'SystolicBP', 'DiastolicBP', 'FastingBloodSugar', 
                    'HbA1c', 'SerumCreatinine', 'HemoglobinLevels', 'CholesterolTotal']

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
axes = axes.flatten()

for idx, col in enumerate(unified_features):
    ax = axes[idx]
    sns.boxplot(x='Diagnosis', y=col, data=df, ax=ax, palette=['#2ecc71', '#e74c3c'])
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_xticklabels(['No CKD', 'CKD'])

plt.suptitle('Unified Schema Features - CKD Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}/07_unified_schema_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary Statistics
print('='*70)
print('CHRONIC KIDNEY DISEASE DATASET SUMMARY')
print('='*70)
print(f'Total Samples: {len(df)}')
print(f'Total Features: {len(df.columns)}')
print(f'\nClass Distribution:')
print(f'  - No CKD: {(df["Diagnosis"]==0).sum()} ({(df["Diagnosis"]==0).sum()/len(df)*100:.1f}%)')
print(f'  - CKD: {(df["Diagnosis"]==1).sum()} ({(df["Diagnosis"]==1).sum()/len(df)*100:.1f}%)')
print(f'  - Class Ratio: 11.29:1 (SEVERE IMBALANCE)')
print(f'\nMissing Values: {df.isnull().sum().sum()}')
print(f'\nFeatures Mapping to Unified Schema: 15')
print(f'\nKey Actions Needed:')
print('  1. SMOTE oversampling for minority class')
print('  2. Use class weights in training')
print('  3. Focus on Precision-Recall metrics')
print('='*70)

In [ ]:
# List saved figures
print('\nSaved figures:')
for f in sorted(os.listdir(FIGURES_PATH)):
    print(f'  - {f}')